**QUBO Check:**
Before applying the formulation to real data, the cells below verify the algebra on a small 3-asset example (N=3,K=2) where we chose to pick 2 out of the 3 assets, and its small enough to brute force all 2^N possible combinations.

In [1]:
import numpy as np

In [2]:
# example using 3 assets
mu=np.array([0.10,0.12,0.08])
sigma= np.array([[0.04,0.01,0.00]
                ,[0.01,0.05,0.02],
                [0.00,0.02,0.03]])
N,K = 3,2
theta1,theta2=1.0,1.0


We have 3 assets with their expected returns (mu) and how they move together (sigma), we chose to select 2 assets, and 2 weighting knobs (theta1=risk term, theta2=return term) which we borrow from the mean-variance objective (w^TCw - q*mu^Tw) and set to 1.0 for simplicity. This simply declares the problem

In [8]:
#penaltu bound for safety margin
A_min = np.max(np.abs(theta1*sigma)) + theta2*np.max(np.abs(mu))
A= 3*A_min # saftey margin. tuned empirically when real data is used



We computed the penalty coeeficient of A. Keeping in mind that if A is too small, the optimizer breaks and chooses wrong number of assets as its easier than obeying the given constraint. A_min becaome sthe floor where the optimizer stops breaking (we multiply by 3 just for safety since our bound is still relatively small).

In [6]:
Q = theta1 *sigma.copy() # for 01z^TCz term (which is already quadratic so sigma's numbers->Q)

for i in range(N):
    Q[i,i] += -theta2 * mu[i] + A * (1-2*K)
    for j in range(N):
        if i != j:
            Q[i,j] += A


In [10]:
# brute force every z in {0,1}^3 and check that the QUBO min matches (2^3=8 possible asset selections)
#gives the best (sum(z)=K) portfolio
best_z,best_val = None, np.inf
for combo in range(2**N):
    z=np.array([(combo>>i) & 1 for i in range(N)])
    val = z @ Q @ z        # computes H(z)=z^TQz which is the QUBO cost for the chosen combination portfolio
    if val < best_val:
        best_val,best_z = val,z # verifies whichever z produces smallest H(z) and its sum = to assets chosen (K=2)

print("Q=",Q)
print("Optimal z:", best_z, "H(z) =",round(best_val,3), " feasible (sum=K)?", best_z.sum()==K)

Q= [[-1.59  0.52  0.51]
 [ 0.52 -1.6   0.53]
 [ 0.51  0.53 -1.58]]
Optimal z: [1 0 1] H(z) = -2.15  feasible (sum=K)? True
